# INT8 quantization benchmark

INT8 runs faster by using 8-bit arithmetic, and that can cost accuracy. This notebook
measures the cost directly, reporting mAP next to latency for every model it quantises,
since a speed number on its own does not say whether the model is still worth running.

TensorRT learns the INT8 ranges from a calibration set. Those calibration frames come
from the training split, and accuracy is measured on a separate held-out set of
validation frames, so the quantisation is never tuned to the images it is graded on.

This study runs at a square 640 input. Ultralytics can only validate an engine on square
images, and the point here is the drop from FP16 to INT8 measured the same way for both,
which a square input gives cleanly. The rectangular 224 by 640 latency lives in the other
notebook.

## How to run this one
The TensorRT and ModelOpt packages replace several core libraries, so the runtime has to
restart once for them to load cleanly. Press **Run all**. Partway through it restarts
itself; when it does, press **Run all again**. The second pass skips the restart and
continues.

When prompted, upload `data/int8_calib.zip` and the two `.pt` files from `models/`. The
last cell downloads `int8_results.json` for the repo's `reports/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Install everything the engine export needs up front. Doing it here, rather than letting
# Ultralytics auto-install mid-export, keeps the package swap in one place so a single
# restart afterwards leaves a consistent environment.
!pip install -q ultralytics "tensorrt-cu12" onnxruntime-gpu onnxslim "nvidia-modelopt[onnx]"

In [ ]:
# Restart once so the freshly installed torch and CUDA libraries load cleanly. The flag
# file survives the restart, so the second Run all pass skips this and carries on instead
# of restarting again.
import os

FLAG = '/content/_int8_restarted'
if not os.path.exists(FLAG):
    open(FLAG, 'w').close()
    print('Restarting the runtime to load the updated packages. Press Run all again.')
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
else:
    print('runtime already restarted, continuing')

In [ ]:
import tensorrt as trt
import ultralytics

print('ultralytics', ultralytics.__version__)
print('tensorrt', trt.__version__)

In [ ]:
# Upload the calibration bundle and both checkpoints, unzip, and point the two yamls at
# their absolute location, which is only known here on Colab.
import zipfile
from pathlib import Path

import yaml
from google.colab import files

uploaded = files.upload()

zips = [n for n in uploaded if n.endswith('.zip')]
assert zips, 'upload int8_calib.zip'
with zipfile.ZipFile(zips[0]) as z:
    z.extractall('.')

root = Path('int8_calib').resolve()
for name in ('calib.yaml', 'eval.yaml'):
    conf = yaml.safe_load((root / name).read_text())
    conf['path'] = str(root)
    (root / name).write_text(yaml.safe_dump(conf, sort_keys=False))

pt_files = sorted(n for n in uploaded if n.endswith('.pt'))
assert pt_files, 'upload the two .pt files too'
print('checkpoints:', pt_files)
print('calibration frames:', len(list((root / 'images/train').glob('*.jpg'))))
print('held-out eval frames:', len(list((root / 'images/val').glob('*.jpg'))))

In [ ]:
import shutil
import time
from statistics import median

import numpy as np
from ultralytics import YOLO

IMGSZ = 640
FRAMES = [np.random.randint(0, 255, (375, 1242, 3), dtype=np.uint8) for _ in range(60)]
CALIB_YAML = str(root / 'calib.yaml')
EVAL_YAML = str(root / 'eval.yaml')


def time_model(model, warmup=10):
    """End-to-end latency over predict, summarised by the median."""
    for _ in range(warmup):
        model.predict(FRAMES[0], imgsz=IMGSZ, device=0, verbose=False)
    latencies = []
    for frame in FRAMES:
        start = time.perf_counter()
        model.predict(frame, imgsz=IMGSZ, device=0, verbose=False)
        latencies.append((time.perf_counter() - start) * 1000.0)
    med = median(sorted(latencies))
    return {'median_ms': round(med, 3), 'fps': round(1000.0 / med, 1) if med else 0.0}


def build_and_measure(pt, precision):
    """Export one engine, score it on the held-out eval, and time it."""
    kwargs = dict(format='engine', imgsz=IMGSZ, device=0, verbose=False)
    if precision == 'int8':
        kwargs.update(int8=True, data=CALIB_YAML)  # calibrate on the training frames
    else:
        kwargs.update(half=True)
    engine = YOLO(pt).export(**kwargs)
    # Rename so the FP16 engine is not overwritten by the INT8 build of the same model.
    dst = f'{Path(pt).stem}_{precision}.engine'
    shutil.move(engine, dst)

    model = YOLO(dst, task='detect')
    acc = model.val(data=EVAL_YAML, imgsz=IMGSZ, device=0, verbose=False, plots=False)
    latency = time_model(model)
    return {
        'mAP50': round(float(acc.box.map50), 4),
        'mAP50_95': round(float(acc.box.map), 4),
        **latency,
    }


print('helpers ready')

In [ ]:
# Build FP16 then INT8 for each model. The INT8 build runs a calibration pass first, so it
# takes longer than the FP16 one.
gpu_name = !nvidia-smi --query-gpu=name --format=csv,noheader
results = {
    'gpu': gpu_name[0].strip(),
    'imgsz': IMGSZ,
    'eval_frames': len(list((root / 'images/val').glob('*.jpg'))),
    'models': {},
}

for pt in pt_files:
    name = Path(pt).stem
    results['models'][name] = {}
    for precision in ('fp16', 'int8'):
        print(f'building {name} {precision} ...')
        stats = build_and_measure(pt, precision)
        results['models'][name][precision] = stats
        print(f'  mAP50 {stats["mAP50"]}   {stats["median_ms"]} ms   {stats["fps"]} FPS')

print('\ndone')

In [ ]:
# Table with the accuracy cost and latency gain of INT8 over FP16.
print(f"GPU: {results['gpu']}   accuracy on {results['eval_frames']} held-out frames, square {IMGSZ}\n")
print(f"{'model':16s} {'prec':6s} {'mAP50':>7s} {'median ms':>11s} {'FPS':>8s}")
for model, prec in results['models'].items():
    for name, stats in prec.items():
        print(f"{model:16s} {name:6s} {stats['mAP50']:>7.3f} {stats['median_ms']:>11.3f} {stats['fps']:>8.1f}")
    if 'fp16' in prec and 'int8' in prec:
        dmap = prec['int8']['mAP50'] - prec['fp16']['mAP50']
        speedup = prec['fp16']['median_ms'] / prec['int8']['median_ms']
        print(f"  -> INT8 costs {dmap:+.3f} mAP50 and runs {speedup:.2f}x faster than FP16\n")

In [ ]:
import json

with open('int8_results.json', 'w') as f:
    json.dump(results, f, indent=2)

from google.colab import files

files.download('int8_results.json')

## Back in the repo

Move `int8_results.json` to `reports/` and commit it. The final results step reads the
FP16 and INT8 accuracy and latency from here and reports INT8 as its trade against FP16:
how much mAP it costs for how much extra speed.

Two things to keep in mind when reading it. Accuracy is on a 128-frame held-out subset,
so the INT8 column is the drop from FP16 on the same frames, not a full-set score. And
the input is square 640 here, so these latency numbers sit beside the rectangular 224 by
640 table rather than inside it.